# Biblioteca `category_encoders`

Este notebook mostra como usar a biblioteca `category_encoders` para transformar variáveis categóricas em variáveis numéricas, uma etapa essencial no pré-processamento de dados para modelos de Machine Learning.

## Instalação

Se a biblioteca ainda não estiver instalada, execute:

```
pip install category_encoders
```

In [1]:
import pandas as pd
import category_encoders as ce

print("Versão do category_encoders:", ce.__version__)

Versão do category_encoders: 2.10.0


## Criando um dataset de exemplo

Vamos criar um pequeno dataset fictício com variáveis categóricas para demonstrar os diferentes tipos de encoders.

In [2]:
dados = pd.DataFrame({
    'cidade': ['São Paulo', 'Rio de Janeiro', 'Belo Horizonte', 'São Paulo', 'Curitiba', 'Rio de Janeiro', 'Curitiba'],
    'nivel_educacao': ['Fundamental', 'Médio', 'Superior', 'Superior', 'Médio', 'Fundamental', 'Superior'],
    'sexo': ['F', 'M', 'F', 'M', 'F', 'M', 'F'],
    'salario': [1500, 2500, 4200, 3800, 2200, 1800, 5000]
})

# Variável alvo (target) usada pelos encoders supervisionados
target = pd.Series([0, 1, 1, 1, 0, 0, 1], name='comprou_produto')

dados

,cidade,nivel_educacao,sexo,salario
0,São Paulo,Fundamental,F,1500
1,Rio de Janeiro,Médio,M,2500
2,Belo Horizonte,Superior,F,4200
3,São Paulo,Superior,M,3800
4,Curitiba,Médio,F,2200
5,Rio de Janeiro,Fundamental,M,1800
6,Curitiba,Superior,F,5000


## 1. OneHotEncoder

Cria uma nova coluna binária (0/1) para cada categoria. Ideal para variáveis nominais com poucas categorias, mas pode gerar muitas colunas se houver muitas categorias (alta cardinalidade).

In [3]:
onehot_encoder = ce.OneHotEncoder(cols=['cidade'], use_cat_names=True)
dados_onehot = onehot_encoder.fit_transform(dados)
dados_onehot

,cidade_São Paulo,cidade_Rio de Janeiro,cidade_Belo Horizonte,cidade_Curitiba,nivel_educacao,sexo,salario
0,1,0,0,0,Fundamental,F,1500
1,0,1,0,0,Médio,M,2500
2,0,0,1,0,Superior,F,4200
3,1,0,0,0,Superior,M,3800
4,0,0,0,1,Médio,F,2200
5,0,1,0,0,Fundamental,M,1800
6,0,0,0,1,Superior,F,5000


## 2. OrdinalEncoder

Atribui um número inteiro a cada categoria. Útil quando existe uma ordem natural entre as categorias (ex.: nível de educação).

In [4]:
# É possível definir manualmente a ordem das categorias
ordem_educacao = [{
    'col': 'nivel_educacao',
    'mapping': {'Fundamental': 1, 'Médio': 2, 'Superior': 3}
}]

ordinal_encoder = ce.OrdinalEncoder(cols=['nivel_educacao'], mapping=ordem_educacao)
dados_ordinal = ordinal_encoder.fit_transform(dados)
dados_ordinal

,cidade,nivel_educacao,sexo,salario
0,São Paulo,1,F,1500
1,Rio de Janeiro,2,M,2500
2,Belo Horizonte,3,F,4200
3,São Paulo,3,M,3800
4,Curitiba,2,F,2200
5,Rio de Janeiro,1,M,1800
6,Curitiba,3,F,5000


## 3. BinaryEncoder

Converte cada categoria em um número inteiro e depois representa esse número em binário, distribuindo os bits em várias colunas. Gera menos colunas que o OneHotEncoder, sendo mais eficiente para variáveis com muitas categorias.

In [5]:
binary_encoder = ce.BinaryEncoder(cols=['cidade'])
dados_binary = binary_encoder.fit_transform(dados)
dados_binary

,cidade_0,cidade_1,cidade_2,nivel_educacao,sexo,salario
0,0,0,1,Fundamental,F,1500
1,0,1,0,Médio,M,2500
2,0,1,1,Superior,F,4200
3,0,0,1,Superior,M,3800
4,1,0,0,Médio,F,2200
5,0,1,0,Fundamental,M,1800
6,1,0,0,Superior,F,5000


## 4. TargetEncoder

Substitui cada categoria pela média da variável alvo (target) associada a ela. É um encoder supervisionado, ou seja, precisa da variável alvo durante o `fit`. Muito útil para variáveis de alta cardinalidade, mas requer cuidado com vazamento de dados (data leakage) — normalmente aplicado apenas no conjunto de treino.

In [6]:
target_encoder = ce.TargetEncoder(cols=['cidade'])
dados_target = target_encoder.fit_transform(dados['cidade'], target)
dados_target

,cidade
0,0.561296
1,0.561296
2,0.627189
3,0.561296
4,0.561296
5,0.561296
6,0.561296


## 5. HashingEncoder

Usa uma função hash para mapear categorias em um número fixo de colunas. Muito útil quando não se sabe de antemão todas as categorias possíveis (ex.: dados em streaming) ou quando a cardinalidade é muito alta.

In [7]:
hashing_encoder = ce.HashingEncoder(cols=['cidade'], n_components=4)
dados_hashing = hashing_encoder.fit_transform(dados)
dados_hashing

,col_0,col_1,col_2,col_3,nivel_educacao,sexo,salario
0,0,0,0,1,Fundamental,F,1500
1,1,0,0,0,Médio,M,2500
2,1,0,0,0,Superior,F,4200
3,0,0,0,1,Superior,M,3800
4,0,0,1,0,Médio,F,2200
5,1,0,0,0,Fundamental,M,1800
6,0,0,1,0,Superior,F,5000


## 6. CountEncoder

Substitui cada categoria pela quantidade (frequência) de vezes que ela aparece no dataset. Simples e rápido, útil como feature adicional em modelos baseados em árvore.

In [8]:
count_encoder = ce.CountEncoder(cols=['cidade'])
dados_count = count_encoder.fit_transform(dados)
dados_count

,cidade,nivel_educacao,sexo,salario
0,2,Fundamental,F,1500
1,2,Médio,M,2500
2,1,Superior,F,4200
3,2,Superior,M,3800
4,2,Médio,F,2200
5,2,Fundamental,M,1800
6,2,Superior,F,5000


### ⚠️ Limitação do CountEncoder: colisão de categorias

Repare que **São Paulo** e **Rio de Janeiro** aparecem 2 vezes cada uma, então ambas recebem o mesmo valor codificado (`2`). Isso faz com que o modelo **não consiga distinguir** essas duas cidades apenas por essa coluna — elas passam a ser tratadas como se tivessem a mesma "importância" numérica, o que normalmente não é o desejado (a menos que a frequência em si seja de fato a informação relevante).

Duas formas de contornar esse problema:

1. **Usar a contagem apenas como feature auxiliar**, mantendo a identidade da categoria com outro encoder (ex.: OneHot ou Ordinal) em paralelo — assim nenhuma informação é perdida.
2. **Usar um encoder supervisionado que combine estatística com o histórico da categoria**, como o `CatBoostEncoder` — ele calcula a média do target por categoria usando uma técnica *ordered target statistics* (parecida com leave-one-out), reduzindo bastante a chance de duas categorias diferentes ficarem com o mesmo valor.

In [9]:
# Opção 1: manter a contagem apenas como feature auxiliar,
# preservando a identidade da categoria com um OrdinalEncoder em paralelo
count_encoder = ce.CountEncoder(cols=['cidade'])
ordinal_encoder_aux = ce.OrdinalEncoder(cols=['cidade'])

dados_combinado = dados.copy()
dados_combinado['cidade_id'] = ordinal_encoder_aux.fit_transform(dados[['cidade']])['cidade']
dados_combinado['cidade_freq'] = count_encoder.fit_transform(dados[['cidade']])['cidade']

dados_combinado[['cidade', 'cidade_id', 'cidade_freq']]

,cidade,cidade_id,cidade_freq
0,São Paulo,1,2
1,Rio de Janeiro,2,2
2,Belo Horizonte,3,1
3,São Paulo,1,2
4,Curitiba,4,2
5,Rio de Janeiro,2,2
6,Curitiba,4,2


In [10]:
# Opção 2: CatBoostEncoder — encoder supervisionado que reduz a colisão,
# pois combina o histórico do target por categoria (ordered target statistics,
# semelhante a leave-one-out). Nota: a 1ª ocorrência de cada categoria ainda
# recebe o valor "prior" (média global), mas ocorrências seguintes já
# refletem o comportamento específico daquela categoria em relação ao target.
catboost_encoder = ce.CatBoostEncoder(cols=['cidade'])
dados_catboost = catboost_encoder.fit_transform(dados['cidade'], target)

pd.concat([dados['cidade'], dados_catboost.rename(columns={'cidade': 'cidade_catboost'})], axis=1)

,cidade,cidade_catboost
0,São Paulo,0.571429
1,Rio de Janeiro,0.571429
2,Belo Horizonte,0.571429
3,São Paulo,0.285714
4,Curitiba,0.571429
5,Rio de Janeiro,0.785714
6,Curitiba,0.285714


## Combinando encoders em um Pipeline do scikit-learn

Os encoders do `category_encoders` seguem a interface do scikit-learn (`fit`, `transform`, `fit_transform`), por isso podem ser usados diretamente em um `Pipeline`.

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ('encoder', ce.TargetEncoder(cols=['cidade', 'nivel_educacao', 'sexo'])),
    ('modelo', LogisticRegression())
])

pipeline.fit(dados, target)
print("Predições:", pipeline.predict(dados))

Predições: [0 1 1 1 0 0 1]


## Galeria completa: todos os encoders do `category_encoders`

Abaixo, um exemplo de cada encoder disponível na biblioteca, aplicado simultaneamente às colunas `cidade`, `sexo` e `nivel_educacao` do dataset `dados`.

- **Não supervisionados** — usam apenas `encoder.fit(dados)`, sem depender de `target`.
- **Supervisionados** — precisam do `target` no `fit`, pois calculam estatísticas relacionadas à variável a ser prevista (`encoder.fit(dados, target)`).

> ⚠️ `GLMMEncoder` depende do pacote `statsmodels` (`pip install statsmodels`). `WOEEncoder` exige que o `target` seja binário (0/1), o que já é o caso aqui.

In [12]:
# BackwardDifferenceEncoder (não supervisionado) — compara cada nível com o nível anterior
encoder = ce.BackwardDifferenceEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade_0,cidade_1,cidade_2,nivel_educacao_0,nivel_educacao_1,sexo_0,salario
0,-0.75,-0.5,-0.25,-0.666667,-0.333333,-0.5,1500
1,0.25,-0.5,-0.25,0.333333,-0.333333,0.5,2500
2,0.25,0.5,-0.25,0.333333,0.666667,-0.5,4200
3,-0.75,-0.5,-0.25,0.333333,0.666667,0.5,3800
4,0.25,0.5,0.75,0.333333,-0.333333,-0.5,2200
5,0.25,-0.5,-0.25,-0.666667,-0.333333,0.5,1800
6,0.25,0.5,0.75,0.333333,0.666667,-0.5,5000


In [13]:
# BaseNEncoder (não supervisionado) — generaliza o BinaryEncoder para qualquer base numérica
encoder = ce.BaseNEncoder(cols=['cidade', 'sexo', 'nivel_educacao'], base=3)
encoder.fit(dados)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade_0,cidade_1,nivel_educacao_0,nivel_educacao_1,sexo_0,salario
0,0,1,0,1,1,1500
1,0,2,0,2,2,2500
2,1,0,1,0,1,4200
3,0,1,1,0,2,3800
4,1,1,0,2,1,2200
5,0,2,0,1,2,1800
6,1,1,1,0,1,5000


In [14]:
# BinaryEncoder (não supervisionado)
encoder = ce.BinaryEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade_0,cidade_1,cidade_2,nivel_educacao_0,nivel_educacao_1,sexo_0,sexo_1,salario
0,0,0,1,0,1,0,1,1500
1,0,1,0,1,0,1,0,2500
2,0,1,1,1,1,0,1,4200
3,0,0,1,1,1,1,0,3800
4,1,0,0,1,0,0,1,2200
5,0,1,0,0,1,1,0,1800
6,1,0,0,1,1,0,1,5000


In [15]:
# CatBoostEncoder (supervisionado) — usa estatística ordenada do target
encoder = ce.CatBoostEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados, target)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade,nivel_educacao,sexo,salario
0,0.523810,0.190476,0.514286,1500
1,0.523810,0.523810,0.642857,2500
2,0.571429,0.892857,0.514286,4200
3,0.523810,0.892857,0.642857,3800
4,0.523810,0.523810,0.514286,2200
5,0.523810,0.190476,0.642857,1800
6,0.523810,0.892857,0.514286,5000


In [32]:
target

0    0
1    1
2    1
3    1
4    0
5    0
6    1
Name: comprou_produto, dtype: int64

In [16]:
# CountEncoder (não supervisionado) — frequência de cada categoria
encoder = ce.CountEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade,nivel_educacao,sexo,salario
0,2,2,4,1500
1,2,2,3,2500
2,1,3,4,4200
3,2,3,3,3800
4,2,2,4,2200
5,2,2,3,1800
6,2,3,4,5000


In [17]:
# GLMMEncoder (supervisionado) — usa um modelo linear generalizado misto (requer statsmodels)
encoder = ce.GLMMEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados, target)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade,nivel_educacao,sexo,salario
0,-0.039465,-1.292270,-0.060765,1500
1,-0.039466,-0.071554,0.083767,2500
2,0.145855,1.452852,-0.060765,4200
3,-0.039465,1.452852,0.083767,3800
4,-0.039465,-0.071554,-0.060765,2200
5,-0.039466,-1.292270,0.083767,1800
6,-0.039465,1.452852,-0.060765,5000


In [18]:
# GrayEncoder (não supervisionado) — código Gray: apenas 1 bit muda entre categorias consecutivas
encoder = ce.GrayEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade_0,cidade_1,cidade_2,nivel_educacao_0,nivel_educacao_1,sexo_0,sexo_1,salario
0,1,1,0,0,1,0,1,1500
1,0,1,0,1,1,1,1,2500
2,0,0,1,1,0,0,1,4200
3,1,1,0,1,0,1,1,3800
4,0,1,1,1,1,0,1,2200
5,0,1,0,0,1,1,1,1800
6,0,1,1,1,0,0,1,5000


In [19]:
# HashingEncoder (não supervisionado)
encoder = ce.HashingEncoder(cols=['cidade', 'sexo', 'nivel_educacao'], n_components=8)
encoder.fit(dados)
X_cleaned = encoder.transform(dados)
X_cleaned

,col_0,col_1,col_2,col_3,col_4,col_5,col_6,col_7,salario
0,0,0,1,1,0,0,0,1,1500
1,2,0,0,0,1,0,0,0,2500
2,1,0,1,1,0,0,0,0,4200
3,0,0,0,2,1,0,0,0,3800
4,1,0,2,0,0,0,0,0,2200
5,1,0,0,0,1,0,0,1,1800
6,0,0,2,1,0,0,0,0,5000


In [20]:
# HelmertEncoder (não supervisionado) — compara cada nível com a média dos níveis subsequentes
encoder = ce.HelmertEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade_0,cidade_1,cidade_2,nivel_educacao_0,nivel_educacao_1,sexo_0,salario
0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1500
1,1.0,-1.0,-1.0,1.0,-1.0,1.0,2500
2,0.0,2.0,-1.0,0.0,2.0,-1.0,4200
3,-1.0,-1.0,-1.0,0.0,2.0,1.0,3800
4,0.0,0.0,3.0,1.0,-1.0,-1.0,2200
5,1.0,-1.0,-1.0,-1.0,-1.0,1.0,1800
6,0.0,0.0,3.0,0.0,2.0,-1.0,5000


In [21]:
# JamesSteinEncoder (supervisionado) — encolhe a média da categoria em direção à média global
encoder = ce.JamesSteinEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados, target)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade,nivel_educacao,sexo,salario
0,0.515152,0.0,0.500000,1500
1,0.515152,0.5,0.666667,2500
2,1.000000,1.0,0.500000,4200
3,0.515152,1.0,0.666667,3800
4,0.515152,0.5,0.500000,2200
5,0.515152,0.0,0.666667,1800
6,0.515152,1.0,0.500000,5000


In [22]:
# LeaveOneOutEncoder (supervisionado) — média do target excluindo a própria linha
encoder = ce.LeaveOneOutEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados, target)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade,nivel_educacao,sexo,salario
0,0.500000,0.0,0.500000,1500
1,0.500000,0.5,0.666667,2500
2,0.571429,1.0,0.500000,4200
3,0.500000,1.0,0.666667,3800
4,0.500000,0.5,0.500000,2200
5,0.500000,0.0,0.666667,1800
6,0.500000,1.0,0.500000,5000


In [23]:
# MEstimateEncoder (supervisionado) — versão simplificada do TargetEncoder com suavização
encoder = ce.MEstimateEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados, target)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade,nivel_educacao,sexo,salario
0,0.523810,0.190476,0.514286,1500
1,0.523810,0.523810,0.642857,2500
2,0.785714,0.892857,0.514286,4200
3,0.523810,0.892857,0.642857,3800
4,0.523810,0.523810,0.514286,2200
5,0.523810,0.190476,0.642857,1800
6,0.523810,0.892857,0.514286,5000


In [24]:
# OneHotEncoder (não supervisionado)
encoder = ce.OneHotEncoder(cols=['cidade', 'sexo', 'nivel_educacao'], use_cat_names=True)
encoder.fit(dados)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade_São Paulo,cidade_Rio de Janeiro,cidade_Belo Horizonte,cidade_Curitiba,nivel_educacao_Fundamental,nivel_educacao_Médio,nivel_educacao_Superior,sexo_F,sexo_M,salario
0,1,0,0,0,1,0,0,1,0,1500
1,0,1,0,0,0,1,0,0,1,2500
2,0,0,1,0,0,0,1,1,0,4200
3,1,0,0,0,0,0,1,0,1,3800
4,0,0,0,1,0,1,0,1,0,2200
5,0,1,0,0,1,0,0,0,1,1800
6,0,0,0,1,0,0,1,1,0,5000


In [25]:
# OrdinalEncoder (não supervisionado) — sem ordem customizada, usa a ordem de aparição das categorias
encoder = ce.OrdinalEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade,nivel_educacao,sexo,salario
0,1,1,1,1500
1,2,2,2,2500
2,3,3,1,4200
3,1,3,2,3800
4,4,2,1,2200
5,2,1,2,1800
6,4,3,1,5000


In [26]:
# PolynomialEncoder (não supervisionado) — assume categorias ordinais e cria contrastes polinomiais
encoder = ce.PolynomialEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade_0,cidade_1,cidade_2,nivel_educacao_0,nivel_educacao_1,sexo_0,salario
0,-0.670820,0.5,-0.223607,-7.071068e-01,0.408248,-0.707107,1500
1,-0.223607,-0.5,0.670820,-5.551115e-17,-0.816497,0.707107,2500
2,0.223607,-0.5,-0.670820,7.071068e-01,0.408248,-0.707107,4200
3,-0.670820,0.5,-0.223607,7.071068e-01,0.408248,0.707107,3800
4,0.670820,0.5,0.223607,-5.551115e-17,-0.816497,-0.707107,2200
5,-0.223607,-0.5,0.670820,-7.071068e-01,0.408248,0.707107,1800
6,0.670820,0.5,0.223607,7.071068e-01,0.408248,-0.707107,5000


In [27]:
# QuantileEncoder (supervisionado) — codifica pelo quantil (padrão: mediana) do target por categoria
encoder = ce.QuantileEncoder(cols=['cidade', 'sexo', 'nivel_educacao'], quantile=0.5)
encoder.fit(dados, target)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade,nivel_educacao,sexo,salario
0,0.666667,0.333333,0.6,1500
1,0.666667,0.666667,1.0,2500
2,1.000000,1.000000,0.6,4200
3,0.666667,1.000000,1.0,3800
4,0.666667,0.666667,0.6,2200
5,0.666667,0.333333,1.0,1800
6,0.666667,1.000000,0.6,5000


In [28]:
# RankHotEncoder (não supervisionado) — "one-hot acumulado", preserva noção de ordem entre categorias
encoder = ce.RankHotEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade_1,cidade_2,cidade_3,cidade_4,nivel_educacao_1,nivel_educacao_2,nivel_educacao_3,sexo_1,sexo_2,salario
0,1,1,1,1,1,0,0,1,0,1500
1,1,1,1,0,1,1,0,1,1,2500
2,1,0,0,0,1,1,1,1,0,4200
3,1,1,1,1,1,1,1,1,1,3800
4,1,1,0,0,1,1,0,1,0,2200
5,1,1,1,0,1,0,0,1,1,1800
6,1,1,0,0,1,1,1,1,0,5000


In [29]:
# SumEncoder (não supervisionado) — compara cada nível com a média geral (contrastes de soma)
encoder = ce.SumEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade_0,cidade_1,cidade_2,nivel_educacao_0,nivel_educacao_1,sexo_0,salario
0,1.0,0.0,0.0,1.0,0.0,1.0,1500
1,0.0,1.0,0.0,0.0,1.0,-1.0,2500
2,0.0,0.0,1.0,-1.0,-1.0,1.0,4200
3,1.0,0.0,0.0,-1.0,-1.0,-1.0,3800
4,-1.0,-1.0,-1.0,0.0,1.0,1.0,2200
5,0.0,1.0,0.0,1.0,0.0,-1.0,1800
6,-1.0,-1.0,-1.0,-1.0,-1.0,1.0,5000


In [30]:
# TargetEncoder (supervisionado)
encoder = ce.TargetEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados, target)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade,nivel_educacao,sexo,salario
0,0.561296,0.490371,0.55943,1500
1,0.561296,0.561296,0.58614,2500
2,0.627189,0.637628,0.55943,4200
3,0.561296,0.637628,0.58614,3800
4,0.561296,0.561296,0.55943,2200
5,0.561296,0.490371,0.58614,1800
6,0.561296,0.637628,0.55943,5000


In [31]:
# WOEEncoder (supervisionado) — Weight of Evidence, exige target binário (0/1)
encoder = ce.WOEEncoder(cols=['cidade', 'sexo', 'nivel_educacao'])
encoder.fit(dados, target)
X_cleaned = encoder.transform(dados)
X_cleaned

,cidade,nivel_educacao,sexo,salario
0,-0.182322,-1.280934,-0.182322,1500
1,-0.182322,-0.182322,0.223144,2500
2,0.000000,1.203973,-0.182322,4200
3,-0.182322,1.203973,0.223144,3800
4,-0.182322,-0.182322,-0.182322,2200
5,-0.182322,-1.280934,0.223144,1800
6,-0.182322,1.203973,-0.182322,5000


## Resumo

| Encoder | Tipo | Quando usar |
|---|---|---|
| OneHotEncoder | Não supervisionado | Poucas categorias, sem ordem |
| OrdinalEncoder | Não supervisionado | Categorias com ordem natural |
| BinaryEncoder | Não supervisionado | Muitas categorias, reduz dimensionalidade |
| TargetEncoder | Supervisionado | Alta cardinalidade, cuidado com vazamento |
| HashingEncoder | Não supervisionado | Cardinalidade muito alta ou desconhecida |
| CountEncoder | Não supervisionado | Feature de frequência; cuidado com colisão entre categorias de mesma frequência |
| CatBoostEncoder | Supervisionado | Alta cardinalidade, reduz colisão e vazamento graças à estatística ordenada |